# BaIP: learning curve (power-law slope), overnight queue

Trains crop448 and resize448 on 25%, 50% and 75% of each fold's training samples. Validation and test samples are exactly those of `crop448_cv_v1` and `resize448_cv_v1` (the 100% points), so all runs are scored on the same 736 test samples. At the end, the analysis cell fits error = a * n^(-alpha) and extrapolates to 736 + 1354 samples.

**Before you start:** Runtime > Change runtime type > **L4 GPU**. Keep this tab open and the laptop awake all night (`caffeinate -d` in a Mac terminal). Without Colab Pro+, a runtime without an open tab can be shut down.

**Run the cells top to bottom, once.** Cell 6 starts the whole queue in the background and returns immediately. Use cell 7 to check progress.

**Estimate:** 8 experiments x 5 folds. About 7 to 9 hours and 14 to 18 compute units on an L4. The order puts the most important runs first (crop 50%, 25%, 75%, then resize), so a short night still gives a usable curve.

**If Colab disconnects:** reconnect to an L4, run cells 1 to 4, then cell 6 again. Finished folds are skipped; the interrupted fold restarts from epoch 1.

Drive space: each fold saves a 30 MB checkpoint, about 1.2 GB in total, in `MyDrive/BaIP/checkpoints/`.

In [ ]:
# 1. Settings
BRANCH = 'exp/power-law-slope'
QUEUE = 'configs/learning_curve/queue.txt'
PATHS = 'configs/paths_colab.yaml'
REPO = '/content/repo'
RESULTS = '/content/drive/MyDrive/BaIP/results'

In [ ]:
# 2. GPU check (must say NVIDIA L4) and Google Drive
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. Code: clone (first time) or update, check out the branch, install the two extra packages
import os
if not os.path.isdir(REPO):
    !git clone -q https://github.com/merijnmolenaar/Textile_Classification_CNN_Merijn.git {REPO}
!cd {REPO} && git fetch -q && git checkout -q {BRANCH} && git pull -q && git log -1 --format='commit %h  %s'
!pip -q install pyyaml openpyxl
os.chdir(REPO)

In [ ]:
# 4. Data check: expect 2208 images in total (any split over Images / Images-2), labels True,
#    and the two 100% runs present in the results folder
import glob, yaml
p = yaml.safe_load(open(PATHS))['data']
n = sum(len(glob.glob(os.path.join(d, '*.jpg'))) for d in p['image_dirs'] if os.path.isdir(d))
print('images:', n, '(expect 2208)')
print('labels:', os.path.exists(p['labels_xlsx']))
for r in ('crop448_cv_v1', 'resize448_cv_v1'):
    print(r, 'present:', os.path.exists(f'{RESULTS}/{r}/cv_sample_predictions.csv'))

In [ ]:
# 5. Smoke test (about 2 minutes): crop 25%, fold 0, 1 epoch of 3 batches, into results/smoke.
#    Check the class table: 'samples' row should read about 124 / 111 / 148. 0% accuracy is expected.
!python scripts/run_cv.py configs/learning_curve/crop448_lc_f025.yaml --paths {PATHS} --folds 0 \
    --set train.epochs=1 --set train.max_batches=3 --set out_dir={RESULTS}/smoke
!python scripts/run_queue.py {QUEUE} --paths {PATHS} --dry-run

In [ ]:
# 6. Start the whole queue in the background. The cell returns at once; the queue keeps running.
#    Running it again while the queue is still busy does nothing (it checks first).
import subprocess, sys
busy = subprocess.run(['pgrep', '-f', 'scripts/run_queue.py'], capture_output=True, text=True).stdout.split()
if busy:
    print('queue already running, pid', busy)
else:
    os.makedirs(f'{RESULTS}/queue_logs', exist_ok=True)
    out = open(f'{RESULTS}/queue_logs/queue_console.log', 'a')
    proc = subprocess.Popen([sys.executable, '-u', 'scripts/run_queue.py', QUEUE, '--paths', PATHS],
                            stdout=out, stderr=subprocess.STDOUT, cwd=REPO, start_new_session=True)
    print('queue started, pid', proc.pid, '| progress: run cell 7')

In [ ]:
# 7. Progress (run whenever you like): finished experiments, current experiment, latest epoch
import pathlib
st = pathlib.Path(f'{RESULTS}/queue_logs/queue_status.txt')
print(st.read_text()[-1500:] if st.exists() else 'no status yet')
hist = sorted(glob.glob(f'{RESULTS}/*448_lc_*/fold*/history.csv'), key=os.path.getmtime)
if hist:
    h = hist[-1]
    last = open(h).read().strip().splitlines()[-1].split(',')
    print(f'\nlatest: {h.split("/")[-3]} {h.split("/")[-2]} | epoch {last[0]} | val sample acc {float(last[6])*100:.1f}% | {float(last[8]):.0f} s/epoch')
print('queue running:', bool(subprocess.run(['pgrep', '-f', 'scripts/run_queue.py'], capture_output=True).stdout))
!nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader

In [ ]:
# 8. When the queue has finished (or with whatever runs are complete): fit the power law
#    Output: RESULTS/analyses/learning_curve/ (csv, fit text, png)
!python scripts/analysis/learning_curve.py {RESULTS}
from IPython.display import Image, display
png = f'{RESULTS}/analyses/learning_curve/learning_curve.png'
if os.path.exists(png):
    display(Image(png))

**When done:** Runtime > **Disconnect and delete runtime**, so it stops using compute units. The results stay on Drive in `MyDrive/BaIP/results/<run>/`, the logs in `MyDrive/BaIP/results/queue_logs/`.